# Task 6: Stacked Area Chart — Cumulative Installs Over Time by Category

**Requirements:**
- Stacked area chart: cumulative installs over time, one color band per category
- Filters:
  - Average rating >= 4.2
  - App name contains **no numbers**
  - Category starts with **T** or **P**
  - Reviews > 1,000
  - Size between 20 MB and 80 MB
- Legend translation: **Travel & Local → French**, **Productivity → Spanish**, **Photography → Japanese**
- Highlight (increase color intensity) any **month** where **any** category's installs grew more than 25% month-over-month
- Display rule: only visible between **4 PM – 6 PM IST**

### Data note (same as Task 4)
This dataset has no real historical install counts — only one snapshot total per app. As before, we approximate a time series using each app's `Last Updated` month, and take the **cumulative sum per category over time**. This is an approximation, not literal historical data, and is flagged again here for clarity.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

## 1. Load and clean the data

In [2]:
df = pd.read_csv('googleplaystore.csv')
df = df.drop_duplicates(subset='App', keep='first')
df = df[~df['Category'].astype(str).str.contains(r'^\d', regex=True, na=False)]

def parse_size(size):
    if pd.isna(size):
        return np.nan
    size = str(size).strip()
    if size == 'Varies with device' or size == '':
        return np.nan
    if size.endswith('M'):
        return float(size[:-1])
    if size.endswith('k') or size.endswith('K'):
        return float(size[:-1]) / 1024.0
    try:
        return float(size)
    except ValueError:
        return np.nan
df['Size_MB'] = df['Size'].apply(parse_size)

df['Installs_Num'] = pd.to_numeric(
    df['Installs'].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False),
    errors='coerce'
)
df['Reviews_Num'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Last_Updated_Date'] = pd.to_datetime(df['Last Updated'], errors='coerce')

df[['App','Category','Size_MB','Rating','Installs_Num','Reviews_Num','Last_Updated_Date']].head()

,App,Category,Size_MB,Rating,Installs_Num,Reviews_Num,Last_Updated_Date
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,19.0,4.1,10000,159,2018-01-07
1,Coloring book moana,ART_AND_DESIGN,14.0,3.9,500000,967,2018-01-15
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,8.7,4.7,5000000,87510,2018-08-01
3,Sketch - Draw & Paint,ART_AND_DESIGN,25.0,4.5,50000000,215644,2018-06-08
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,2.8,4.3,100000,967,2018-06-20


## 2. Apply filters

In [3]:
cats_TP = [c for c in df['Category'].unique() if str(c)[0] in ('T', 'P')]
print('T/P categories present:', cats_TP)

name_mask = ~df['App'].astype(str).str.contains(r'\d', regex=True, na=False)

mask = (
    (df['Rating'] >= 4.2) &
    name_mask &
    df['Category'].isin(cats_TP) &
    (df['Reviews_Num'] > 1000) &
    (df['Size_MB'] >= 20) & (df['Size_MB'] <= 80) &
    df['Last_Updated_Date'].notna()
)

filtered = df[mask].copy()
print(f"Rows after filtering: {len(filtered)}")
filtered['Category'].value_counts()

T/P categories present: ['PHOTOGRAPHY', 'TRAVEL_AND_LOCAL', 'TOOLS', 'PERSONALIZATION', 'PRODUCTIVITY', 'PARENTING']
Rows after filtering: 110


Category
PHOTOGRAPHY         39
TRAVEL_AND_LOCAL    24
TOOLS               17
PRODUCTIVITY        16
PERSONALIZATION     10
PARENTING            4
Name: count, dtype: int64

## 3. Build the monthly cumulative installs trend per category

In [4]:
filtered['Month'] = filtered['Last_Updated_Date'].dt.to_period('M').dt.to_timestamp()

monthly = filtered.groupby(['Category', 'Month'])['Installs_Num'].sum().reset_index()

full_range = pd.date_range(monthly['Month'].min(), monthly['Month'].max(), freq='MS')
pivot = monthly.pivot(index='Month', columns='Category', values='Installs_Num').reindex(full_range).fillna(0)
pivot.index.name = 'Month'

cumulative = pivot.cumsum()
cumulative.tail()

Category,PARENTING,PERSONALIZATION,PHOTOGRAPHY,PRODUCTIVITY,TOOLS,TRAVEL_AND_LOCAL
Month,,,,,,
2018-04-01,100000.0,22000000.0,103000000.0,1100000.0,1050000.0,5050000.0
2018-05-01,10100000.0,22000000.0,114000000.0,1100000.0,11050000.0,5050000.0
2018-06-01,10100000.0,32000000.0,185000000.0,16100000.0,11050000.0,11050000.0
2018-07-01,10700000.0,43500000.0,443000000.0,182100000.0,233650000.0,42150000.0
2018-08-01,10700000.0,63500000.0,969500000.0,787100000.0,353650000.0,77250000.0


## 4. Flag months where ANY category grew more than 25% month-over-month

In [5]:
pct_change = cumulative.pct_change().replace([np.inf, -np.inf], np.nan)
per_category_flags = pct_change > 0.25

# A month is "highlighted" if ANY category exceeded 25% MoM growth that month
month_highlight = per_category_flags.any(axis=1)
print(f"Highlighted months: {month_highlight.sum()} / {len(month_highlight)}")
month_highlight[month_highlight].index.tolist()[:10]

Highlighted months: 9 / 46


[Timestamp('2017-03-01 00:00:00'),
 Timestamp('2017-09-01 00:00:00'),
 Timestamp('2018-01-01 00:00:00'),
 Timestamp('2018-02-01 00:00:00'),
 Timestamp('2018-04-01 00:00:00'),
 Timestamp('2018-05-01 00:00:00'),
 Timestamp('2018-06-01 00:00:00'),
 Timestamp('2018-07-01 00:00:00'),
 Timestamp('2018-08-01 00:00:00')]

## 5. Legend translation

In [6]:
category_translations = {
    'TRAVEL_AND_LOCAL': 'Voyages et local (Travel & Local)',   # French
    'PRODUCTIVITY': 'Productividad (Productivity)',             # Spanish
    'PHOTOGRAPHY': '\u5199\u771f (Photography)'                 # Japanese: Shashin
}

def display_label(cat):
    return category_translations.get(cat, cat)

{c: display_label(c) for c in cumulative.columns}

{'PARENTING': 'PARENTING',
 'PERSONALIZATION': 'PERSONALIZATION',
 'PHOTOGRAPHY': '写真 (Photography)',
 'PRODUCTIVITY': 'Productividad (Productivity)',
 'TOOLS': 'TOOLS',
 'TRAVEL_AND_LOCAL': 'Voyages et local (Travel & Local)'}

## 6. Stacked area chart (Plotly), with highlighted months

In [7]:
palette = ['#4C9AFF', '#FF8C42', '#3DDC97', '#FF6B6B', '#B084F5', '#FFD166']

fig = go.Figure()

for i, cat in enumerate(cumulative.columns):
    fig.add_trace(go.Scatter(
        x=cumulative.index,
        y=cumulative[cat],
        mode='lines',
        name=display_label(cat),
        stackgroup='installs',
        line=dict(width=0.5, color=palette[i % len(palette)]),
        fillcolor=palette[i % len(palette)]
    ))

# Add semi-transparent vertical bands over highlighted months (increased 'intensity')
shapes = []
months = cumulative.index
for i, m in enumerate(months):
    if month_highlight.iloc[i]:
        # highlight a band from the previous month to this month
        start = months[i-1] if i > 0 else m
        shapes.append(dict(
            type='rect', xref='x', yref='paper',
            x0=start, x1=m, y0=0, y1=1,
            fillcolor='white', opacity=0.12, line=dict(width=0)
        ))

fig.update_layout(
    title='Cumulative Installs by Category (Stacked)<br><sup>Lighter vertical bands = a month where any category grew >25% MoM</sup>',
    xaxis_title='Month',
    yaxis_title='Cumulative Installs',
    shapes=shapes,
    legend=dict(orientation='h', y=1.15, x=0.5, xanchor='center'),
    height=620,
    template='plotly_white',
    margin=dict(t=110, b=60)
)

fig.show()

## 7. Export for the combined dashboard

In [8]:
cumulative.to_csv('task6_cumulative_installs.csv')
month_highlight.to_frame('Highlighted').to_csv('task6_month_highlight.csv')
print('Saved: task6_cumulative_installs.csv, task6_month_highlight.csv')

Saved: task6_cumulative_installs.csv, task6_month_highlight.csv


### Note on the 4PM–6PM IST display rule
Same pattern as Tasks 1–5: dashboard-level display rule, implemented with the same IST time-check logic in the combined `dashboard.html`.